In [4]:
import os
import random
import torch
import numpy as np
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
from tqdm import tqdm
from sklearn.decomposition import PCA

###############################################################################
# Arguments
###############################################################################
args = {
    "models": [
        "nlpaueb/sec-bert-base",
        "bert-base-uncased"
    ],
    "datasets": [
        {
            "name": "yelp_review_full",
            "config": None,
            "split": "train",
            "text_column": "text"
        },
        {
            "name": "wikitext",
            "config": "wikitext-2-raw-v1",
            "split": "train",
            "text_column": "text"
        },
        {
            "name": "ag_news",
            "config": None,
            "split": "train",
            "text_column": "text"
        },
    ],
    "max_texts": 1000,
    "batch_size": 64,
    "drift_strengths": [0.0, 0.25, 0.5, 0.75, 1.0],
    "pca_components": 2,
    "output_dir": "results_multidataset",
}

os.makedirs(args["output_dir"], exist_ok=True)

###############################################################################
# Utility Functions
###############################################################################
def batch_generator(data, batch_size=32):
    for i in range(0, len(data), batch_size):
        yield data[i : i + batch_size]

def extract_cls_embeddings(model, tokenizer, texts, device):
    encodings = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    )
    input_ids = encodings["input_ids"].to(device)
    attention_mask = encodings["attention_mask"].to(device)
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
    return cls_embeddings.cpu().numpy()

def introduce_gradual_drift(text_list, fraction_shuffle=0.5):
    import random
    new_texts = []
    for txt in text_list:
        words = txt.split()
        if len(words) < 2:
            new_texts.append(txt)
            continue
        k = int(len(words) * fraction_shuffle)
        if k < 1:
            new_texts.append(txt)
            continue
        indices = list(range(len(words)))
        random.shuffle(indices)
        shuffle_indices = indices[:k]
        to_shuffle = [words[i] for i in shuffle_indices]
        random.shuffle(to_shuffle)
        for i, idx in enumerate(shuffle_indices):
            words[idx] = to_shuffle[i]
        new_texts.append(" ".join(words))
    return new_texts

import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
from sklearn.decomposition import PCA
from tqdm import tqdm
import random

###############################################################################
# Additional Distance Metric Functions
###############################################################################
def euclidean_distance(x, y):
    """
    Euclidean (L2) Distance

    Formula:
        d_{L2}(x, y) = sqrt( sum_i (x_i - y_i)^2 )
    """
    return np.sqrt(np.sum((x - y) ** 2))

def manhattan_distance(x, y):
    """
    Manhattan (L1) Distance

    Formula:
        d_{L1}(x, y) = sum_i |x_i - y_i|
    """
    return np.sum(np.abs(x - y))

def minkowski_distance(x, y, p=3):
    """
    Minkowski Distance (generalization of L1 and L2)

    Formula:
        d_{p}(x, y) = ( sum_i (|x_i - y_i|^p) )^(1/p)

    - p=1 -> Manhattan distance
    - p=2 -> Euclidean distance
    """
    return np.sum(np.abs(x - y) ** p) ** (1.0 / p)

def chebyshev_distance(x, y):
    """
    Chebyshev (L∞) Distance

    Formula:
        d_{∞}(x, y) = max_i( |x_i - y_i| )
    """
    return np.max(np.abs(x - y))

def correlation_distance(x, y):
    """
    Correlation Distance

    Formula:
        d_corr(x, y) = 1 - corr(x, y)

    where corr(x, y) is the Pearson correlation coefficient:
        corr(x, y) = ( (x - mean(x)) · (y - mean(y)) )
                     / ( ||x - mean(x)|| * ||y - mean(y)|| )

    Note: This distance ignores differences in magnitude and focuses on the correlation
          between x and y.
    """
    x_centered = x - np.mean(x)
    y_centered = y - np.mean(y)
    numerator = np.dot(x_centered, y_centered)
    denominator = (np.linalg.norm(x_centered) * np.linalg.norm(y_centered)) + 1e-12
    corr = numerator / denominator
    return 1.0 - corr

def canberra_distance(x, y):
    """
    Canberra Distance

    Formula:
        d_canberra(x, y) = sum_i( |x_i - y_i| / (|x_i| + |y_i|) )

    More sensitive to differences when x_i or y_i are near zero.
    """
    numerator = np.abs(x - y)
    denominator = np.abs(x) + np.abs(y) + 1e-12  # small epsilon to avoid division by zero
    return np.sum(numerator / denominator)


###############################################################################
# Drift Detection Class (using Mahalanobis distance)
###############################################################################
class EmbeddingTracker:
    def __init__(self, embedding_dim, alpha=0.01):
        self.alpha = alpha
        self.mean = np.zeros((embedding_dim,))
        self.cov = np.eye(embedding_dim)
        self.count = 0

    def update(self, embedding):
        if self.count == 0:
            self.mean = embedding
            self.cov = np.eye(len(embedding))
        else:
            self.mean = (1 - self.alpha) * self.mean + self.alpha * embedding
            diff = embedding - self.mean
            self.cov = (1 - self.alpha) * self.cov + self.alpha * np.outer(diff, diff)
        self.count += 1

    def mahalanobis_distance(self, embedding):
        diff = embedding - self.mean
        cov_inv = np.linalg.pinv(self.cov)
        return np.sqrt(diff.T @ cov_inv @ diff)

###############################################################################
# Main Data Collection
###############################################################################
def collect_data():
    if torch.backends.mps.is_available():
        device = torch.device("mps")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")
    print("Using device:", device)

    from collections import defaultdict
    results = defaultdict(list)

    for dataset_info in args["datasets"]:
        dataset_name  = dataset_info["name"]
        dataset_config = dataset_info["config"]
        dataset_split = dataset_info["split"]
        text_col      = dataset_info["text_column"]

        print(f"\n=== Loading dataset: {dataset_name} ===")
        ds = load_dataset(dataset_name, dataset_config, split=dataset_split)
        texts = list(ds[text_col])
        random.shuffle(texts)
        if args["max_texts"] > 0 and len(texts) > args["max_texts"]:
            texts = texts[: args["max_texts"]]

        half_point = len(texts) // 2
        baseline_texts = texts[:half_point]
        drift_texts    = texts[half_point:]

        for model_name in args["models"]:
            print(f"\n--- Using Model: {model_name} ---")
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            model = AutoModel.from_pretrained(model_name)
            model.to(device)
            model.eval()

            baseline_embs = []
            for b in batch_generator(baseline_texts, args["batch_size"]):
                emb_b = extract_cls_embeddings(model, tokenizer, b, device)
                baseline_embs.append(emb_b)
            baseline_embs = np.concatenate(baseline_embs, axis=0)

            pca = PCA(n_components=args["pca_components"])
            pca.fit(baseline_embs)

            key = (dataset_name, model_name)

            for drift_strength in args["drift_strengths"]:
                print(f"Simulating drift_strength={drift_strength} ...")
                drifted_texts = introduce_gradual_drift(drift_texts, fraction_shuffle=drift_strength)
                test_texts = baseline_texts + drifted_texts

                # No PCA
                embedding_dim_no_pca = baseline_embs.shape[1]
                tracker_no_pca = EmbeddingTracker(embedding_dim_no_pca, alpha=0.01)
                distance_scores_no_pca = []
                all_embeddings_no_pca = []

                # Initialize tracker with baseline
                for batch in batch_generator(baseline_texts, args["batch_size"]):
                    emb = extract_cls_embeddings(model, tokenizer, batch, device)
                    mean_emb = emb.mean(axis=0)
                    tracker_no_pca.update(mean_emb)

                for batch in tqdm(batch_generator(test_texts, args["batch_size"]), leave=False):
                    emb = extract_cls_embeddings(model, tokenizer, batch, device)
                    all_embeddings_no_pca.extend(emb)
                    mean_emb = emb.mean(axis=0)
                    dist = tracker_no_pca.mahalanobis_distance(mean_emb)
                    distance_scores_no_pca.append(dist)
                    tracker_no_pca.update(mean_emb)

                final_dist_no_pca = distance_scores_no_pca[-1] if distance_scores_no_pca else 0.0

                # PCA
                embedding_dim_pca = args["pca_components"]
                tracker_pca = EmbeddingTracker(embedding_dim_pca, alpha=0.01)
                distance_scores_pca = []
                all_embeddings_pca = []

                # Initialize tracker with baseline (reduced)
                for batch in batch_generator(baseline_texts, args["batch_size"]):
                    emb = extract_cls_embeddings(model, tokenizer, batch, device)
                    emb_pca = pca.transform(emb)
                    mean_emb = emb_pca.mean(axis=0)
                    tracker_pca.update(mean_emb)

                for batch in tqdm(batch_generator(test_texts, args["batch_size"]), leave=False):
                    emb = extract_cls_embeddings(model, tokenizer, batch, device)
                    emb_pca = pca.transform(emb)
                    all_embeddings_pca.extend(emb_pca)
                    mean_emb = emb_pca.mean(axis=0)
                    dist = tracker_pca.mahalanobis_distance(mean_emb)
                    distance_scores_pca.append(dist)
                    tracker_pca.update(mean_emb)

                final_dist_pca = distance_scores_pca[-1] if distance_scores_pca else 0.0

                # Store with 'final_similarity' key to avoid KeyError in plotting
                results[key].append({
                    "drift_strength": drift_strength,
                    "pca": False,
                    "time_series": distance_scores_no_pca[:],
                    "final_similarity": final_dist_no_pca,
                    "all_embeddings": np.array(all_embeddings_no_pca)
                })
                results[key].append({
                    "drift_strength": drift_strength,
                    "pca": True,
                    "time_series": distance_scores_pca[:],
                    "final_similarity": final_dist_pca,
                    "all_embeddings": np.array(all_embeddings_pca)
                })

    print("\nData collection done!")
    return results

all_results = collect_data()


Using device: mps

=== Loading dataset: yelp_review_full ===

--- Using Model: nlpaueb/sec-bert-base ---
Simulating drift_strength=0.0 ...


Simulating drift_strength=0.25 ...


Simulating drift_strength=0.5 ...


Simulating drift_strength=0.75 ...


Simulating drift_strength=1.0 ...


...

Simulating drift_strength=0.25 ...


Simulating drift_strength=0.5 ...


Simulating drift_strength=0.75 ...


Simulating drift_strength=1.0 ...



Data collection done!


In [5]:
###############################################################################
# 2) Plotting: Reproduce 6 subplots, but using Mahalanobis Distance
###############################################################################
import matplotlib.pyplot as plt
import numpy as np

def plot_six_subplots(all_results):
    for (dataset_name, model_name), runs in all_results.items():
        no_pca = [r for r in runs if r["pca"] is False]
        pca_   = [r for r in runs if r["pca"] is True]

        no_pca = sorted(no_pca, key=lambda x: x["drift_strength"])
        pca_   = sorted(pca_,   key=lambda x: x["drift_strength"])

        x_no_pca = [r["drift_strength"] for r in no_pca]
        y_no_pca = [r["final_similarity"] for r in no_pca]
        x_pca    = [r["drift_strength"] for r in pca_]
        y_pca    = [r["final_similarity"] for r in pca_]

        def rolling_mean_std(values, window=2):
            means, stds = [], []
            for i in range(len(values)):
                wstart = max(0, i - window + 1)
                slice_ = values[wstart:i+1]
                means.append(np.mean(slice_))
                stds.append(np.std(slice_))
            return np.array(means), np.array(stds)

        y_no_pca_rm, y_no_pca_std = rolling_mean_std(y_no_pca, window=2)
        y_pca_rm,    y_pca_std    = rolling_mean_std(y_pca,    window=2)

        all_final_sims_no_pca = y_no_pca
        all_final_sims_pca    = y_pca

        # For largest drift strength
        baseline_run_no_pca = [r for r in no_pca if r["drift_strength"] == 0.0][0]
        drift_run_no_pca    = [r for r in no_pca if r["drift_strength"] == 1.0][0]
        baseline_run_pca    = [r for r in pca_   if r["drift_strength"] == 0.0][0]
        drift_run_pca       = [r for r in pca_   if r["drift_strength"] == 1.0][0]

        fig, axs = plt.subplots(2, 3, figsize=(15,8))
        axs = axs.flatten()

        # 1) Distance vs Drift Strength
        axs[0].plot(x_no_pca, y_no_pca, marker='o', label='No PCA')
        axs[0].plot(x_pca,    y_pca,    marker='s', label='PCA')
        axs[0].set_title("Mahalanobis Distance vs Drift Strength")
        axs[0].set_xlabel("Drift Strength")
        axs[0].set_ylabel("Mahalanobis Distance")
        axs[0].legend()
        axs[0].grid(True, linestyle='--', alpha=0.5)

        # 2) Rolling Mean & Std
        axs[1].plot(x_no_pca, y_no_pca_rm, color='blue', label='No PCA - Mean')
        axs[1].fill_between(x_no_pca,
                            y_no_pca_rm - y_no_pca_std,
                            y_no_pca_rm + y_no_pca_std,
                            color='blue', alpha=0.2)

        axs[1].plot(x_pca, y_pca_rm, color='orange', label='PCA - Mean')
        axs[1].fill_between(x_pca,
                            y_pca_rm - y_pca_std,
                            y_pca_rm + y_pca_std,
                            color='orange', alpha=0.2)
        axs[1].set_title("Rolling Mean & Std of Mahalanobis Distance")
        axs[1].set_xlabel("Drift Strength")
        axs[1].set_ylabel("Mahalanobis Distance")
        axs[1].legend()
        axs[1].grid(True, linestyle='--', alpha=0.5)

        # 3) Histogram of final distances
        bins = np.linspace(min(all_final_sims_no_pca + all_final_sims_pca),
                           max(all_final_sims_no_pca + all_final_sims_pca), 10)
        axs[2].hist(all_final_sims_no_pca, bins=bins, alpha=0.7, label='No PCA')
        axs[2].hist(all_final_sims_pca,    bins=bins, alpha=0.7, label='PCA')
        axs[2].set_title("Histogram of Final Distances")
        axs[2].set_xlabel("Mahalanobis Distance")
        axs[2].set_ylabel("Frequency")
        axs[2].legend()

        # 4) Scatter Plot (largest drift vs baseline) in PCA space
        baseline_embs_2d = baseline_run_pca["all_embeddings"]
        drift_embs_2d    = drift_run_pca["all_embeddings"]
        axs[3].scatter(baseline_embs_2d[:, 0], baseline_embs_2d[:, 1], alpha=0.6, label="Baseline")
        axs[3].scatter(drift_embs_2d[:, 0],    drift_embs_2d[:, 1],    alpha=0.6, label="Drifted")
        axs[3].set_title("Scatter Plot (PCA Space)")
        axs[3].set_xlabel("PC1")
        axs[3].set_ylabel("PC2")
        axs[3].legend()

        # 5) Delta from Baseline Distance
        baseline_dist_no_pca = baseline_run_no_pca["final_similarity"]
        baseline_dist_pca    = baseline_run_pca["final_similarity"]
        delta_no_pca = [dist - baseline_dist_no_pca for dist in y_no_pca]
        delta_pca    = [dist - baseline_dist_pca    for dist in y_pca]
        axs[4].plot(x_no_pca, delta_no_pca, marker='o', label='No PCA')
        axs[4].plot(x_pca,    delta_pca,    marker='s', label='PCA')
        axs[4].axhline(0.0, color='gray', linestyle='--', alpha=0.7)
        axs[4].set_title("Delta from Baseline (Mahalanobis Distance)")
        axs[4].set_xlabel("Drift Strength")
        axs[4].set_ylabel("Delta Distance")
        axs[4].legend()
        axs[4].grid(True, linestyle='--', alpha=0.5)

        # 6) Compare final distances (No PCA vs PCA) directly
        axs[5].scatter(x_no_pca, y_no_pca, color='blue', label='No PCA', alpha=0.6)
        axs[5].scatter(x_pca,    y_pca,    color='orange', label='PCA', alpha=0.6)
        axs[5].set_title("Final Mahalanobis Distances (No PCA vs PCA)")
        axs[5].set_xlabel("Drift Strength")
        axs[5].set_ylabel("Mahalanobis Distance")
        axs[5].legend()
        axs[5].grid(True, linestyle='--', alpha=0.5)

        fig.suptitle(f"{dataset_name} | {model_name}", fontsize=16)
        fig.tight_layout()

        model_name_safe = model_name.replace("/", "_")
        fname = f"{dataset_name}_{model_name_safe}_6subplots_mahalanobis.png"
        save_path = os.path.join(args["output_dir"], fname)
        plt.savefig(save_path)
        plt.show()
        print(f"Saved 6-subplot figure: {save_path}")

plot_six_subplots(all_results)
print("All done.")


Saved 6-subplot figure: results_multidataset/yelp_review_full_nlpaueb_sec-bert-base_6subplots_mahalanobis.png
Saved 6-subplot figure: results_multidataset/yelp_review_full_bert-base-uncased_6subplots_mahalanobis.png
Saved 6-subplot figure: results_multidataset/wikitext_nlpaueb_sec-bert-base_6subplots_mahalanobis.png
Saved 6-subplot figure: results_multidataset/wikitext_bert-base-uncased_6subplots_mahalanobis.png
Saved 6-subplot figure: results_multidataset/ag_news_nlpaueb_sec-bert-base_6subplots_mahalanobis.png
Saved 6-subplot figure: results_multidataset/ag_news_bert-base-uncased_6subplots_mahalanobis.png
All done.
